## Extracting unique columns per participant

In [14]:
import pandas as pd
from pathlib import Path

part_dir = Path("../data/by_participant")

all_cols = set()

for csv_path in sorted(part_dir.glob("participant_*.csv")):
    df = pd.read_csv(csv_path, nrows=5, low_memory=False) 
    all_cols.update(df.columns)

print("Total unique columns:", len(all_cols))
for col in sorted(all_cols):
    print(col)

Total unique columns: 59
AOI Group Left
AOI Group Right
AOI Name Left
AOI Name Right
AOI Order Binocular
AOI Order Right
AOI Scope Left
AOI Scope Right
Annotation Description
Annotation Name
Annotation Tags
Category Group
Category Left
Category Right
Color
Content
Export End Trial Time [ms]
Export Start Trial Time [ms]
Eye Position Left X [mm]
Eye Position Left Y [mm]
Eye Position Left Z [mm]
Eye Position Right X [mm]
Eye Position Right Y [mm]
Eye Position Right Z [mm]
Gaze Vector Left X
Gaze Vector Left Y
Gaze Vector Left Z
Gaze Vector Right X
Gaze Vector Right Y
Gaze Vector Right Z
Index Left
Index Right
Mouse Position X [px]
Mouse Position Y [px]
Participant
Point of Regard Left X [px]
Point of Regard Left Y [px]
Point of Regard Right X [px]
Point of Regard Right Y [px]
Port Status
Pupil Diameter Left [mm]
Pupil Diameter Right [mm]
Pupil Position Left X [px]
Pupil Position Left Y [px]
Pupil Position Right X [px]
Pupil Position Right Y [px]
Pupil Size Left X [px]
Pupil Size Left Y [p

## column mapping dictionary
### building a dictionary to map various column names to standard names

In [15]:
column_map = {
    # participant + meta
    "Participant": "participant",
    "Category Group": "category_group",
    "Stimulus": "stimulus",
    "Trial": "trial",
    "Unnamed: 0": None,
    "groupe d'enfants": None,
    "Color": None,
    "content":None,

    # timing
    "RecordingTime [ms]": "time_ms",
    "Time of Day [h:m:s:ms]": "time_of_day",
    "Export Start Trial Time [ms]": None,
    "Export End Trial Time [ms]": None,
    "index_right": None,
    "index_left": None,

    # AOI
    "AOI Group Left": None,
    "AOI Group Right": None,
    "AOI Name Left": None,
    "AOI Name Right": None,
    "AOI Order Right": None,
    "AOI Order Binocular": None,
    "AOI Scope Left": None,
    "AOI Scope Right": None,

    # pupil diameter
    "Pupil Diameter Left [mm]": "pupil_diameter_left_mm",
    "Pupil Diameter Right [mm]": "pupil_diameter_right_mm",

    # eye position
    "Eye Position Left X [mm]": None,
    "Eye Position Left Y [mm]": None,
    "Eye Position Left Z [mm]": None,
    "Eye Position Right X [mm]": None,
    "Eye Position Right Y [mm]": None,
    "Eye Position Right Z [mm]": None,

    # gaze vectors
    "Gaze Vector Left X": None,
    "Gaze Vector Left Y": None,
    "Gaze Vector Left Z": None,
    "Gaze Vector Right X": None,
    "Gaze Vector Right Y": None,
    "Gaze Vector Right Z": None,

    # point of regard
    "Point of Regard Left X [px]": "point_Of_Regard_left_x_px",
    "Point of Regard Left Y [px]": "point_Of_Regard_left_y_px",
    "Point of Regard Right X [px]": "point_Of_Regard_right_x_px",
    "Point of Regard Right Y [px]": "point_Of_Regard_right_y_px",

    # pupil position (px)
    "Pupil Position Left X [px]": None,
    "Pupil Position Left Y [px]": None,
    "Pupil Position Right X [px]": None,
    "Pupil Position Right Y [px]": None,

    # pupil size (px)
    "Pupil Size Left X [px]": None,
    "Pupil Size Left Y [px]": None,
    "Pupil Size Right X [px]": None,
    "Pupil Size Right Y [px]": None,

    # mouse + scroll
    "Mouse Position X [px]": None,
    "Mouse Position Y [px]": None,
    "Scroll Direction X": None,
    "Scroll Direction Y": None,
    "Port Status": None,

    # annotations
    "Annotation Name": None,
    "Annotation Description": None,
    "Annotation Tags": None,
}

### cleaning per participant data

In [16]:
import pandas as pd
from pathlib import Path

in_dir = Path("../data/by_participant")
out_dir = Path("../data/1_by_participant_standardized")
out_dir.mkdir(exist_ok=True)

for csv_path in sorted(in_dir.glob("participant_*.csv")):
    df = pd.read_csv(csv_path, low_memory=False)

    # rename known columns
    rename_dict = {c: column_map[c] for c in df.columns if c in column_map and column_map[c] is not None}
    df = df.rename(columns=rename_dict)

    # drop columns explicitly mapped to None
    drop_cols = [c for c in df.columns if c in column_map and column_map[c] is None]
    df = df.drop(columns=drop_cols, errors="ignore")

    # lowercase leftover columns that were not mapped
    df.columns = [c.lower().strip().replace(" ", "_") for c in df.columns]

    df.to_csv(out_dir / csv_path.name, index=False)
    print("Standardized:", csv_path.name)

Standardized: participant_1.csv
Standardized: participant_10.csv
Standardized: participant_11.csv
Standardized: participant_13.csv
Standardized: participant_14.csv
Standardized: participant_15.csv
Standardized: participant_17.csv
Standardized: participant_18.csv
Standardized: participant_19.csv
Standardized: participant_2.csv
Standardized: participant_20.csv
Standardized: participant_21.csv
Standardized: participant_22.csv
Standardized: participant_23.csv
Standardized: participant_24.csv
Standardized: participant_25.csv
Standardized: participant_26.csv
Standardized: participant_27.csv
Standardized: participant_28.csv
Standardized: participant_29.csv
Standardized: participant_3.csv
Standardized: participant_30.csv
Standardized: participant_31.csv
Standardized: participant_32.csv
Standardized: participant_33.csv
Standardized: participant_34.csv
Standardized: participant_35.csv
Standardized: participant_36.csv
Standardized: participant_37.csv
Standardized: participant_38.csv
Standardized:

In [17]:
import pandas as pd
from pathlib import Path

# Load all participant data
in_dir = Path("../data/1_by_participant_standardized")
all_files = sorted(in_dir.glob("participant_*.csv"))
df = pd.concat(
    (pd.read_csv(f, low_memory=False) for f in all_files),
    ignore_index=True,
    sort=False
)
# ---- TOTAL missing across all participants ----
total_samples = len(df)

total_right_missing = df["pupil_diameter_right_mm"].isna().sum()
total_left_missing  = df["pupil_diameter_left_mm"].isna().sum()

total_right_missing_pct = 100 * total_right_missing / total_samples
total_left_missing_pct  = 100 * total_left_missing  / total_samples

print("=== Overall missing pupil diameter ===")
print(f"Total samples: {total_samples:,}")
print(f"Right pupil missing: {total_right_missing:,} "
      f"({total_right_missing_pct:.2f}%)")
print(f"Left pupil missing:  {total_left_missing:,} "
      f"({total_left_missing_pct:.2f}%)")

=== Overall missing pupil diameter ===
Total samples: 1,351,157
Right pupil missing: 376,758 (27.88%)
Left pupil missing:  445,638 (32.98%)
